# Exploratory Data Analysis — Income and Life Expectancy, 1952–2007

**Partner B · DSE 511 Homework #2**

This notebook reads the cleaned Gapminder panel produced by `src/clean_data.py`
(`data/processed/gapminder_clean.csv`) and answers one question:
**how are national income and life expectancy related, and how has that relationship changed between 1952 and 2007?**

Part 1 computes four descriptive statistics. Part 2 (next commit) produces the figures.

In [1]:
from pathlib import Path
import subprocess, sys

import numpy as np
import pandas as pd

pd.set_option("display.precision", 2)
pd.set_option("display.width", 120)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

ROOT = Path.cwd().resolve()
if not (ROOT / "data").exists():          # notebook launched from notebooks/
    ROOT = ROOT.parent
CLEAN = ROOT / "data" / "processed" / "gapminder_clean.csv"

# Re-create the cleaned file from the raw data if it is not present yet.
if not CLEAN.exists():
    subprocess.run([sys.executable, str(ROOT / "src" / "clean_data.py")], check=True)

df = pd.read_csv(CLEAN)
df["continent"] = df["continent"].astype("category")
print(f"{df.shape[0]} rows x {df.shape[1]} cols, "
      f"{df['country'].nunique()} countries, {df['year'].min()}–{df['year'].max()}")
df.head()

1704 rows x 8 cols, 142 countries, 1952–2007


,country,continent,year,life_expectancy,population,gdp_per_capita,gdp_total_bn,log_gdp_per_capita
0,Afghanistan,Asia,1952,28.801,8425333,779.445,6.567,2.892
1,Afghanistan,Asia,1957,30.332,9240934,820.853,7.585,2.914
2,Afghanistan,Asia,1962,31.997,10267083,853.101,8.759,2.931
3,Afghanistan,Asia,1967,34.020,11537966,836.197,9.648,2.922
4,Afghanistan,Asia,1972,36.088,13079460,739.981,9.679,2.869


## Part 1 — Descriptive statistics

### Statistic 1: Summary of the numeric variables

`describe()` gives the centre, spread and range of every numeric column across all 1,704 country-years.
Note the gap between mean and median GDP per capita: income is strongly right-skewed.

In [2]:
numeric = ["life_expectancy", "gdp_per_capita", "population", "gdp_total_bn"]
summary = df[numeric].describe().T
summary["skew"] = df[numeric].skew()
summary

,count,mean,std,min,25%,50%,75%,max,skew
life_expectancy,"1,704.000",59.474,12.917,23.599,48.198,60.712,70.846,82.603,-0.253
gdp_per_capita,"1,704.000","7,215.327","9,857.455",241.166,"1,202.060","3,531.847","9,325.462","113,523.133",3.850
population,"1,704.000","29,601,212.325","106,157,896.744","60,011.000","2,793,664.000","7,023,595.500","19,585,221.750","1,318,683,096.000",8.340
gdp_total_bn,"1,704.000",186.810,714.030,0.053,5.895,22.343,105.744,"12,934.459",10.248


### Statistic 2: How the world changed, 1952 vs 2007

Mean and median of life expectancy and GDP per capita in the first and last year of the panel,
and the change between them (the same 142 countries appear in both years).

In [3]:
ends = df[df["year"].isin([1952, 2007])]
change = (ends.groupby("year")[["life_expectancy", "gdp_per_capita"]]
              .agg(["mean", "median", "min", "max"]))
change.loc["change 1952→2007"] = change.loc[2007] - change.loc[1952]
change.round(1)

life_expectancy                      gdp_per_capita                              
                            mean median    min    max           mean    median     min         max
year                                                                                              
1952                      49.100 45.100 28.800 72.700      3,725.300 1,968.500 298.800 108,382.400
2007                      67.000 71.900 39.600 82.600     11,680.100 6,124.400 277.600  49,357.200
change 1952→2007          17.900 26.800 10.800  9.900      7,954.800 4,155.800 -21.300 -59,025.200

### Statistic 3: Continent means in 2007

Average life expectancy and income by continent in the most recent year, with the number of countries in each group.

In [4]:
by_cont = (df[df["year"] == 2007]
           .groupby("continent", observed=True)
           .agg(countries=("country", "size"),
                mean_life_expectancy=("life_expectancy", "mean"),
                median_life_expectancy=("life_expectancy", "median"),
                mean_gdp_per_capita=("gdp_per_capita", "mean"),
                median_gdp_per_capita=("gdp_per_capita", "median"))
           .sort_values("mean_life_expectancy", ascending=False))
by_cont.round(1)

,countries,mean_life_expectancy,median_life_expectancy,mean_gdp_per_capita,median_gdp_per_capita
continent,,,,,
Oceania,2,80.700,80.700,"29,810.200","29,810.200"
Europe,30,77.600,78.600,"25,054.500","28,054.100"
Americas,25,73.600,72.900,"11,003.000","8,948.100"
Asia,33,70.700,72.400,"12,473.000","4,471.100"
Africa,52,54.800,52.900,"3,089.000","1,452.300"


### Statistic 4: Correlation between income and life expectancy

Pearson correlation on the raw dollar scale versus the log₁₀ scale, plus the rank-based Spearman
correlation (which is identical on both scales because log is monotonic). The log scale captures the
relationship much better: doubling income buys roughly the same extra years whether you start poor or rich.

In [5]:
corr = pd.DataFrame({
    "Pearson r": [df["life_expectancy"].corr(df["gdp_per_capita"]),
                  df["life_expectancy"].corr(df["log_gdp_per_capita"])],
    "Spearman ρ": [df["life_expectancy"].corr(df["gdp_per_capita"], method="spearman"),
                   df["life_expectancy"].corr(df["log_gdp_per_capita"], method="spearman")],
}, index=["gdp_per_capita (dollars)", "log10(gdp_per_capita)"])
corr.round(3)

,Pearson r,Spearman ρ
gdp_per_capita (dollars),0.584,0.826
log10(gdp_per_capita),0.808,0.826


The correlation is not constant over time. Below is the Pearson correlation between life expectancy
and log income within each year of the panel.

In [6]:
corr_by_year = (df.groupby("year")
                  .apply(lambda g: g["life_expectancy"].corr(g["log_gdp_per_capita"]), include_groups=False)
                  .rename("r(life_expectancy, log10 gdp_per_capita)"))
corr_by_year.round(3).to_frame().T

year,1952,1957,1962,1967,1972,1977,1982,1987,1992,1997,2002,2007
"r(life_expectancy, log10 gdp_per_capita)",0.748,0.759,0.771,0.773,0.789,0.814,0.846,0.874,0.856,0.864,0.825,0.809
